<a href="https://colab.research.google.com/github/Addreta/Subgraph-Centrality/blob/main/qsvt_sc_protein_RINs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pennylane biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 6.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 7.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 11.5 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 6.9 MB/s  0:00:01
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [pennylane]


In [ ]:
#this cell holds the whole ideal-circuit pipeline as functions, so the same steps can be run on any graph.
#the steps are: exact SC (numpy) -> normalization and Chebyshev polynomials -> the two ideal QSVT circuits -> ranking comparison.
#bipartite graphs have no odd closed walks, so for them only the even circuit is run (the sinh part is exactly zero).
import numpy as np, networkx as nx, pandas as pd, time, warnings
import pennylane as qml
from scipy.linalg import expm
from scipy.special import ive
from scipy.stats import spearmanr, kendalltau
from numpy.polynomial import chebyshev as C
warnings.filterwarnings("ignore")

def tie_classes(sc, tol=1e-9):                    # vertices whose SC agree to 1e-9 (relative) are one class
    order = np.argsort(-sc)
    cls = np.zeros(len(sc), dtype=int)
    for a, b in zip(order[:-1], order[1:]):
        cls[b] = cls[a] + int(abs(sc[a] - sc[b]) > tol * abs(sc[a]))
    return cls                                    # class 0 = most central

def cheb_coeffs(z, deg, parity):
    k = np.arange(deg + 1)
    c = 2.0 * ive(k, z)
    c[0] = ive(0, z)
    c[k % 2 != parity] = 0.0
    if parity == 0:
        return c / (0.5 * (1 + np.exp(-2 * z)))    # divide by cosh(z), scaled by e^-z to match ive
    return c / (0.5 * (1 - np.exp(-2 * z)))        # divide by sinh(z), scaled by e^-z to match ive

def tail_error(z, deg, parity):
    return float(np.sum(np.abs(cheb_coeffs(z, deg + 60, parity)[deg + 1:])))

def degree_for_eps(z, eps, parity, max_deg=300):
    d = parity
    while tail_error(z, d, parity) > eps:
        d += 2
        if d > max_deg:
            raise RuntimeError("tolerance not reached; raise max_deg")
    return d, cheb_coeffs(z, d, parity)

def compare(sc_exact, sc_est, k):
    cls = tie_classes(sc_exact)
    if cls.max() == 0:                            # all SC equal (e.g. K5): a ranking of identical values is undefined
        return dict(rho=np.nan, tau=np.nan, overlap=np.nan)
    est = np.array(sc_est, dtype=float); ex = np.array(sc_exact, dtype=float)
    for c in np.unique(cls):
        est[cls == c] = est[cls == c].mean()      # same value inside a tie class
        ex[cls == c] = ex[cls == c].mean()
    top_ex = set(np.argsort(-ex, kind="stable")[:k]); top_est = set(np.argsort(-est, kind="stable")[:k])
    return dict(rho=spearmanr(ex, est)[0], tau=kendalltau(ex, est)[0], overlap=len(top_ex & top_est) / k)

def run_graph(G, beta=1.0, weighted=False):
    t0 = time.time()
    # ---- exact reference (numpy) and the networkx check ----
    nodes = sorted(G.nodes()); n = len(nodes)
    q = int(np.ceil(np.log2(n))); N = 2 ** q                       # padding: add isolated dummy vertices up to 2^q
    A = nx.to_numpy_array(G, nodelist=nodes, weight="weight" if weighted else None)
    A_p = np.zeros((N, N)); A_p[:n, :n] = A
    SC_exact = np.diag(expm(beta * A))
    nx_diff = np.nan
    if beta == 1.0 and not weighted:                              # networkx's subgraph_centrality is unweighted
        sc_nx = nx.subgraph_centrality(G)
        nx_diff = max(abs(sc_nx[v] - SC_exact[i]) for i, v in enumerate(nodes))
    bip = nx.is_bipartite(G)
    # ---- normalization and polynomials ----
    lam = np.max(np.abs(np.linalg.eigvalsh(A)))
    alpha = 1.001 * max(lam, np.sqrt(np.max(np.abs(A @ A).sum(axis=1))))
    z = beta * alpha
    s = np.sort(SC_exact)[::-1]
    gaps = np.abs(np.diff(s)); gaps = gaps[gaps > 1e-9 * s[:-1]]
    abs_gap = gaps.min() if len(gaps) else 0.01 * SC_exact.mean()  # all SC equal: no gap exists, so aim for 1% accuracy
    d_e, c_e = degree_for_eps(z, 0.25 * abs_gap / np.cosh(z), 0)
    d_o, c_o = (None, None) if bip else degree_for_eps(z, 0.25 * abs_gap / np.sinh(z), 1)
    # ---- the ideal circuits ----
    At = A_p / alpha
    be_wires = range(q + 1); rp = q + 1
    BE = qml.BlockEncode(At, wires=be_wires)
    dev = qml.device("default.qubit", wires=q + 2)
    def make_circuit(c):
        angles = qml.poly_to_angles(C.cheb2poly(c), "QSVT")
        pos = [qml.PCPhase(a,  dim=N, wires=be_wires) for a in angles]
        neg = [qml.PCPhase(-a, dim=N, wires=be_wires) for a in angles]
        @qml.qnode(dev)
        def circ(v):
            qml.BasisState(np.array([int(b) for b in format(v, f"0{q}b")]), wires=range(1, 1 + q))
            qml.Hadamard(wires=rp)
            qml.ctrl(qml.QSVT(BE, pos), control=rp, control_values=[0])
            qml.ctrl(qml.QSVT(BE, neg), control=rp, control_values=[1])
            qml.Hadamard(wires=rp)
            return qml.probs(wires=range(q + 2))
        return circ
    def p_success(circ, v):
        bits = [0] + [int(b) for b in format(v, f"0{q}b")] + [0]
        return circ(v)[int("".join(map(str, bits)), 2)]
    circ_e = make_circuit(c_e)
    p_e = np.array([p_success(circ_e, v) for v in range(n)])
    if bip:
        p_o = np.zeros(n)
    else:
        circ_o = make_circuit(c_o)
        p_o = np.array([p_success(circ_o, v) for v in range(n)])
    SC_qsvt = np.cosh(z) * np.sqrt(p_e) + np.sinh(z) * np.sqrt(p_o)
    # ---- comparison ----
    k = 3 if n <= 8 else 5
    m = compare(SC_exact, SC_qsvt, k)
    return dict(n=n, edges=G.number_of_edges(), qubits=q + 2, bipartite=bip, classes=int(tie_classes(SC_exact).max() + 1),
                alpha=alpha, deg_even=d_e, deg_odd=d_o, pe_min=p_e.min(), po_min=(np.nan if bip else p_o.min()),
                nx_diff=nx_diff, MAE=np.mean(np.abs(SC_qsvt - SC_exact)), max_rel_err=np.max(np.abs(SC_qsvt - SC_exact) / SC_exact),
                rho=m["rho"], tau=m["tau"], topk=k, overlap=m["overlap"], seconds=time.time() - t0)

In [ ]:
# ---- RIN builder following the procedure of the RinQ paper (2025): C-alpha atoms, 8 A cutoff, unweighted ----
from Bio import PDB
from Bio.PDB import PDBList

def build_rinq(pdb_id, cutoff=8.0):
    path = PDBList().retrieve_pdb_file(pdb_id, file_format="pdb", pdir=".", overwrite=False)
    structure = PDB.PDBParser(QUIET=True).get_structure(pdb_id, path)
    G = nx.Graph()
    for chain in structure.get_chains():            # all chains of all models; nodes are identified by residue number
        for ri in chain:
            for rj in chain:
                if ri != rj and "CA" in ri and "CA" in rj:
                    if np.linalg.norm(ri["CA"].coord - rj["CA"].coord) < cutoff:
                        G.add_edge(ri.get_id()[1], rj.get_id()[1])
    return G

# number of residues stated in the RinQ paper, kept for comparison with the built graphs
rinq_residues = {"1XY1": 10, "4D5M": 11, "2N08": 12, "6A5J": 13, "6RQS": 18, "1Q71": 21,
                 "2MLT": 27, "1A7F": 29, "1GCN": 29, "2K6O": 37, "1JL9": 51, "1UBQ": 76}

In [ ]:
# ---- run all 12 RinQ proteins: ideal QSVT SC (beta = 1) vs classical SC (the Estrada centrality) ----
rows = {}
for pid, n_paper in rinq_residues.items():
    try:
        G = build_rinq(pid)
        r = run_graph(G, beta=1.0, weighted=False)
        m = round(r["overlap"] * r["topk"])
        r["jaccard"] = m / (2 * r["topk"] - m)                  # Jaccard@k from the number of shared top-k vertices
        r["n_paper"] = n_paper
        rows[pid] = r
        print(f"{pid}: n={r['n']} (paper {n_paper}), qubits={r['qubits']}, rho={r['rho']:.3f}, tau={r['tau']:.3f}, "
              f"overlap@{r['topk']}={r['overlap']:.2f}, jaccard={r['jaccard']:.2f}  ({r['seconds']:.0f} s)")
    except Exception as e:
        print(f"{pid}: FAILED -> {type(e).__name__}: {e}")

tab_rin = pd.DataFrame(rows).T[["n_paper", "n", "edges", "qubits", "classes", "deg_even", "deg_odd", "pe_min", "po_min",
                                "nx_diff", "MAE", "max_rel_err", "rho", "tau", "topk", "overlap", "jaccard"]]
fmt = {"pe_min": "{:.1e}".format, "po_min": "{:.1e}".format, "nx_diff": "{:.1e}".format, "MAE": "{:.1e}".format,
       "max_rel_err": "{:.1e}".format, "rho": "{:.3f}".format, "tau": "{:.3f}".format,
       "overlap": "{:.2f}".format, "jaccard": "{:.2f}".format}
print(tab_rin.to_string(formatters=fmt))

Structure exists: './pdb1xy1.ent' 
1XY1: n=9 (paper 10), qubits=6, rho=1.000, tau=1.000, overlap@5=1.00, jaccard=1.00  (1 s)
Structure exists: './pdb4d5m.ent' 
4D5M: n=10 (paper 11), qubits=6, rho=1.000, tau=1.000, overlap@5=1.00, jaccard=1.00  (1 s)
Structure exists: './pdb2n08.ent' 
2N08: n=11 (paper 12), qubits=6, rho=1.000, tau=1.000, overlap@5=1.00, jaccard=1.00  (1 s)
Structure exists: './pdb6a5j.ent' 
6A5J: n=13 (paper 13), qubits=6, rho=1.000, tau=1.000, overlap@5=1.00, jaccard=1.00  (1 s)
Structure exists: './pdb6rqs.ent' 
6RQS: n=16 (paper 18), qubits=6, rho=1.000, tau=1.000, overlap@5=1.00, jaccard=1.00  (2 s)
Structure exists: './pdb1q71.ent' 
1Q71: n=21 (paper 21), qubits=7, rho=1.000, tau=1.000, overlap@5=1.00, jaccard=1.00  (4 s)
Structure exists: './pdb2mlt.ent' 
2MLT: n=26 (paper 27), qubits=7, rho=1.000, tau=1.000, overlap@5=1.00, jaccard=1.00  (3 s)
Structure exists: './pdb1a7f.ent' 
1A7F: n=29 (paper 29), qubits=7, rho=1.000, tau=1.000, overlap@5=1.00, jaccard=1.00 

In [ ]:
# ---- compare graph size with the residues present in each structure file ----
def residue_report(pdb_id):
    st = PDB.PDBParser(QUIET=True).get_structure(pdb_id, f"./pdb{pdb_id.lower()}.ent")
    first = st[0]
    ids_ca = {r.get_id()[1] for c in first for r in c if "CA" in r}
    G = build_rinq(pdb_id)
    return dict(models=len(st), chains=len(first), CA_residues=len(ids_ca),
                graph_nodes=G.number_of_nodes(), no_contact=sorted(ids_ca - set(G.nodes())))

print(pd.DataFrame({p: residue_report(p) for p in rinq_residues}).T.to_string())

Structure exists: './pdb1xy1.ent' 
Structure exists: './pdb4d5m.ent' 
Structure exists: './pdb2n08.ent' 
Structure exists: './pdb6a5j.ent' 
Structure exists: './pdb6rqs.ent' 
Structure exists: './pdb1q71.ent' 
Structure exists: './pdb2mlt.ent' 
Structure exists: './pdb1a7f.ent' 
Structure exists: './pdb1gcn.ent' 
Structure exists: './pdb2k6o.ent' 
Structure exists: './pdb1jl9.ent' 
Structure exists: './pdb1ubq.ent' 
     models chains CA_residues graph_nodes no_contact
1XY1      1      2           9           9         []
4D5M      1      4          10          10         []
2N08     20      1          11          11         []
6A5J     20      1          13          13         []
6RQS     10      1          16          16         []
1Q71     20      1          21          21         []
2MLT      1      2          26          26         []
1A7F     20      2          29          29         []
1GCN      1      1          29          29         []
2K6O      4      1          37          

In [ ]:
# ---- classical SC top-5 of each graph vs the Estrada top-5 listed in Table 2 of the RinQ paper ----
from scipy.linalg import expm

rinq_table2 = {"1A7F": [11, 6, 12, 14, 10], "1GCN": [23, 19, 20, 18, 17], "1JL9": [14, 42, 33, 31, 18],
               "1Q71": [19, 18, 7, 8, 20],  "1UBQ": [5, 4, 23, 3, 6],     "1XY1": [6, 5, 1, 3, 2],
               "2K6O": [29, 13, 10, 18, 14], "2MLT": [12, 9, 13, 8, 14],  "2N08": [6, 5, 7, 8, 4],
               "4D5M": [5, 3, 6, 8, 4],      "6A5J": [8, 9, 6, 7, 5],     "6RQS": [10, 11, 9, 8, 12]}

out = {}
for pid, ref in rinq_table2.items():
    G = build_rinq(pid)
    nodes = sorted(G.nodes())
    sc = np.diag(expm(nx.to_numpy_array(G, nodelist=nodes)))
    top = [nodes[i] for i in np.argsort(-sc, kind="stable")[:5]]
    shared = len(set(top) & set(ref))
    out[pid] = dict(ours=top, paper=ref, shared=shared, jaccard=round(shared / (10 - shared), 2), same_order=(top == ref))
print(pd.DataFrame(out).T.to_string())

Structure exists: './pdb1a7f.ent' 
Structure exists: './pdb1gcn.ent' 
Structure exists: './pdb1jl9.ent' 
Structure exists: './pdb1q71.ent' 
Structure exists: './pdb1ubq.ent' 
Structure exists: './pdb1xy1.ent' 
Structure exists: './pdb2k6o.ent' 
Structure exists: './pdb2mlt.ent' 
Structure exists: './pdb2n08.ent' 
Structure exists: './pdb4d5m.ent' 
Structure exists: './pdb6a5j.ent' 
Structure exists: './pdb6rqs.ent' 
                      ours                 paper shared jaccard same_order
1A7F   [11, 6, 12, 14, 10]   [11, 6, 12, 14, 10]      5     1.0       True
1GCN  [23, 19, 20, 18, 17]  [23, 19, 20, 18, 17]      5     1.0       True
1JL9  [14, 42, 33, 31, 18]  [14, 42, 33, 31, 18]      5     1.0       True
1Q71    [19, 18, 7, 8, 20]    [19, 18, 7, 8, 20]      5     1.0       True
1UBQ      [5, 4, 23, 3, 6]      [5, 4, 23, 3, 6]      5     1.0       True
1XY1       [6, 5, 1, 3, 2]       [6, 5, 1, 3, 2]      5     1.0       True
2K6O  [29, 13, 10, 18, 14]  [29, 13, 10, 18, 14]      5